<a href="https://colab.research.google.com/github/parshav42/-ai-enhanced-intrusion-detection-system/blob/main/sugarcanedatanew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!unzip -b /content/drive/MyDrive/Sugarcane_leafs1.zip -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (368).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (371).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (374).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (38).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (384).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (386).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (392).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (4).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (401).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (403).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (408).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (412).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust (424).jpeg  
  inflating: /content/Sugarcane_leafs/Rust/cropped_rust

In [3]:
import torch
from torch import nn
from torchvision import transforms ,datasets
import os
import sklearn
from sklearn.model_selection import train_test_split
import shutil

In [4]:
import os
import shutil
from sklearn.model_selection import train_test_split

datasets = "/content/Sugarcane_leafs"
output = "/content/Sugarcane"

for folder in os.listdir(datasets):

    files = os.listdir(f"{datasets}/{folder}")

    train, test = train_test_split(
        files,
        test_size=0.2,
        random_state=42
    )

    os.makedirs(f"{output}/train/{folder}", exist_ok=True)
    os.makedirs(f"{output}/test/{folder}", exist_ok=True)

    # Copy train images
    for file in train:
        shutil.copy(
            f"{datasets}/{folder}/{file}",
            f"{output}/train/{folder}/{file}"
        )

    # Copy test images
    for file in test:
        shutil.copy(
            f"{datasets}/{folder}/{file}",
            f"{output}/test/{folder}/{file}"
        )

In [5]:
import os
from PIL import Image

dataset = "/content/Sugarcane/train"

for folder in os.listdir(dataset):

    folder_path = os.path.join(dataset, folder)

    for file in os.listdir(folder_path):

        if file.lower().endswith((".jpg", ".jpeg")):

            image_path = os.path.join(folder_path, file)

            img = Image.open(image_path)

            png_path = os.path.splitext(image_path)[0] + ".png"

            img.save(png_path, "PNG")

            os.remove(image_path)

In [6]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()


])

In [7]:
from torchvision import datasets
train_dataset = datasets.ImageFolder(
    root = '/content/Sugarcane/train', transform = transform

)
test_dataset =datasets.ImageFolder(
    root = '/content/Sugarcane/test', transform = transform

)

In [8]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader( train_dataset , batch_size = 8 ,shuffle = True, )
test_dataloader = DataLoader(test_dataset , batch_size =8, shuffle = True)



In [9]:
from torchvision import models

# class sugarcane(nn.Module):
#   def __init__(self , in_channel , out_channel):
#     super(). __init__()

#     module = nn.Sequential()
model = models.resnet18(weights="DEFAULT")
model.fc = nn.Linear(
    in_features=model.fc.in_features,
    out_features=6
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 199MB/s]


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
for images, labels in train_dataloader:

    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)

    loss = criterion(outputs, labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

In [13]:
torch.save(model.state_dict(), "sugarcane_model.pth")

In [14]:
from torchvision import models
import torch.nn as nn

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 6)   # 6 classes

model.load_state_dict(torch.load("sugarcane_model.pth"))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [27]:
image = Image.open("/content/Sugarcane/test/RedRot/cropped_redrot (109).jpeg").convert("RGB")

image = transform(image)

image = image.unsqueeze(0)

In [28]:
with torch.no_grad():

    output = model(image)

    prediction = output.argmax(1)

print(prediction)

tensor([3])
